# Rag_pipeline_**Member1**

In [ ]:
# ── Upload WikiBio dataset to Colab runtime ───────────────────
#
from google.colab import files
import shutil, os

if not os.path.exists("wikibio_gpt4o.json"):
    print("📤 Please upload wikibio_gpt4o.json ...")
    uploaded = files.upload()
    print("✅ Uploaded:", list(uploaded.keys()))
else:
    print("✅ wikibio_gpt4o.json already present")

embedding generation and storing it to faiss DB

In [ ]:
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from groq import Groq
from datasets import load_dataset

groq_client = Groq(api_key=GROQ_API_KEY)

# ── Global Setup ──────────────────────────────────────────────
print("🔢 Loading embedding model (multi-qa-MiniLM-L6-cos-v1)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    model_kwargs={"device": "cpu"}
)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
BATCH = 500

# ══════════════════════════════════════════════════════════════
#  DATABASE 1: SQuAD (General Knowledge)
# ══════════════════════════════════════════════════════════════
print("\n📥 Loading SQuAD dataset from HuggingFace...")
squad = load_dataset("squad", split="validation")

# We extract 200 unique contexts to keep the index fast and lightweight for the demo
unique_squad_contexts = list(set(squad["context"]))[:200]
squad_docs = []

for ctx in unique_squad_contexts:
    for chunk in splitter.split_text(ctx):
        squad_docs.append(Document(page_content=chunk, metadata={"source": "squad"}))

print(f"📄 {len(squad_docs)} SQuAD chunks created.")
print("🏗️ Building SQuAD FAISS index...")

squad_vectorstore = None
for i in range(0, len(squad_docs), BATCH):
    batch = squad_docs[i:i+BATCH]
    if squad_vectorstore is None:
        squad_vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        squad_vectorstore.merge_from(FAISS.from_documents(batch, embeddings))
    print(f"  Processed {min(i+BATCH, len(squad_docs))}/{len(squad_docs)} SQuAD chunks")

squad_vectorstore.save_local("squad_faiss_index")
print("✅ SQuAD FAISS index saved to 'squad_faiss_index'")


# ══════════════════════════════════════════════════════════════
#  DATABASE 2: WikiBio (Biographies)
# ══════════════════════════════════════════════════════════════
print("\n📥 Loading WikiBio GPT-4o dataset...")
try:
    with open("wikibio_gpt4o.json", "r") as f:
        wikibio_data = json.load(f)
    print(f"✅ Loaded {len(wikibio_data)} WikiBio items")

    wikibio_docs = []
    for item in wikibio_data:
        # Use the first 5 reference samples per item as retrieval corpus
        for sample in item["text_samples"][:5]:
            for chunk in splitter.split_text(sample):
                wikibio_docs.append(Document(
                    page_content=chunk,
                    metadata={"unique_id": item.get("unique_id", ""), "source": "wikibio"}
                ))

    print(f"📄 {len(wikibio_docs)} WikiBio chunks created.")
    print("🏗️ Building WikiBio FAISS index...")

    wikibio_vectorstore = None
    for i in range(0, len(wikibio_docs), BATCH):
        batch = wikibio_docs[i:i+BATCH]
        if wikibio_vectorstore is None:
            wikibio_vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            wikibio_vectorstore.merge_from(FAISS.from_documents(batch, embeddings))
        print(f"  Processed {min(i+BATCH, len(wikibio_docs))}/{len(wikibio_docs)} WikiBio chunks")

    wikibio_vectorstore.save_local("wikibio_faiss_index")
    print("✅ WikiBio FAISS index saved to 'wikibio_faiss_index'")

except FileNotFoundError:
    print("⚠️ ERROR: 'wikibio_gpt4o.json' not found. Please upload it to your Colab files to build the WikiBio index.")


def retriever_agent(question: str, k: int = 5) -> str:
    results = vectorstore.similarity_search(question, k=k)
    return "\n\n".join(f"[Passage {i+1}]\n{d.page_content}" for i, d in enumerate(results))


def generator_agent(question: str, context: str) -> str:
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": """You are a helpful answering agent.
Answer the question completely and concisely.
Use the provided context as your primary source."""},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"}
        ],
        temperature=0.7,
        max_tokens=200
    )
    return resp.choices[0].message.content.strip()


print("✅ Member 1 module ready")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  MEMBER 2 MODULE — Hallucination Detection (HDM)
# ══════════════════════════════════════════════════════════════
import json
import re
import time
import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("📦 Loading sentence-transformer (multi-qa-MiniLM-L6-cos-v1)...")
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")
print("✅ Embedding model loaded")


print("📦 Loading NLI model...")
nli_model = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=-1  # CPU
)
print("✅ NLI model loaded")


def nli_score(context: str, claim: str) -> float:
    """
    Returns a hallucination signal: 1.0 if context CONTRADICTS claim,
    0.5 if NEUTRAL, 0.0 if ENTAILS.
    Truncates to 512 tokens to avoid model limits.
    """
    try:
        # Use first 800 chars of context to stay within token limit
        result = nli_model(
            f"{context[:800]} [SEP] {claim}",
            truncation=True,
            max_length=512
        )[0]
        label = result["label"].upper()
        if label == "CONTRADICTION":
            return 1.0
        elif label == "NEUTRAL":
            return 0.5
        else:  # ENTAILMENT
            return 0.0
    except Exception:
        return 0.5  # neutral fallback

TRIPLET_PROMPT = """Extract factual claims from the text below as Subject-Predicate-Object triplets.
You MUST return ONLY a raw JSON array. No keys like "results", no markdown, no explanation.
Start your response with [ and end with ].

Example output:
[{{"subject": "einstein", "predicate": "developed", "object": "general relativity"}}]

TEXT TO PROCESS:
{text}"""


def extract_triplets(client, text, retries=2):
    for attempt in range(retries):
        try:
            r = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": "You are a JSON generator. You only output valid JSON arrays. Never output anything else."},
                    {"role": "user",   "content": TRIPLET_PROMPT.format(text=text[:1500])}
                ],
                temperature=0.0,
                max_tokens=1024
            )
            raw = r.choices[0].message.content.strip()
            raw = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`")

            # Try direct JSON parse first
            try:
                parsed = json.loads(raw)
                if isinstance(parsed, dict):
                    for key in ["results", "triplets", "data", "output"]:
                        if key in parsed and isinstance(parsed[key], list):
                            parsed = parsed[key]
                            break
                    else:
                        parsed = list(parsed.values())[0] if parsed else []
                if isinstance(parsed, list):
                    valid = [
                        {"subject":   str(t["subject"]).lower().strip(),
                         "predicate": str(t["predicate"]).lower().strip(),
                         "object":    str(t["object"]).lower().strip()}
                        for t in parsed
                        if isinstance(t, dict) and all(k in t for k in ["subject", "predicate", "object"])
                    ]
                    if valid:
                        return valid
            except json.JSONDecodeError:
                pass

            # Fallback: regex to find array
            m = re.search(r"\[.*\]", raw, re.DOTALL)
            if not m:
                print(f"Warning: No JSON array found (attempt {attempt+1})")
                continue
            parsed = json.loads(m.group(0))
            return [
                {"subject":   str(t["subject"]).lower().strip(),
                 "predicate": str(t["predicate"]).lower().strip(),
                 "object":    str(t["object"]).lower().strip()}
                for t in parsed
                if isinstance(t, dict) and all(k in t for k in ["subject", "predicate", "object"])
            ]

        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err:
                wait = 30
                print(f"⏳ Rate limit hit — waiting {wait}s...")
                time.sleep(wait)
                continue
            print(f"Warning: Triplet extraction failed. Error: {e}")
            return []
    return []


def build_kg(triplets: list, name="KG") -> nx.DiGraph:
    G = nx.DiGraph(name=name)
    for t in triplets:
        s, p, o = t["subject"].lower(), t["predicate"].lower(), t["object"].lower()
        G.add_node(s)
        G.add_node(o)
        G.add_edge(s, o, predicate=p)
    return G


def cosine_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na and nb else 0.0

def hallucination_detector(client, context: str, answer: str,
                           threshold: float = 0.30) -> dict:
    """
    Hybrid scorer: KG cosine similarity + NLI contradiction signal.
    score = 0.6 * (1 - cosine_sim) + 0.4 * nli_score
    """
    ctx_triplets = extract_triplets(client, context[:1500])
    ans_triplets = extract_triplets(client, answer[:800])

    print(f"  Triplets — context: {len(ctx_triplets)}, answer: {len(ans_triplets)}")

    G_ctx = build_kg(ctx_triplets, "Context_KG")
    G_ans = build_kg(ans_triplets, "Answer_KG")

    # Save context edges list so we can access the (cu, cv) entities for the penalty
    ctx_edges_list = list(G_ctx.edges(data=True))
    ctx_texts = [f"{u} {d['predicate']} {v}" for u, v, d in ctx_edges_list]
    ctx_embs  = embedding_model.encode(ctx_texts) if ctx_texts else []

    supported, unsupported = [], []

    for u, v, data in G_ans.edges(data=True):
        edge_text = f"{u} {data['predicate']} {v}"

        # ── Signal 1: KG cosine similarity WITH STRICT ENTITY PENALTY ──
        best_cos, best_match = 0.0, ""
        if ctx_texts:
            emb = embedding_model.encode(edge_text)
            for ce, ct, (cu, cv, cdata) in zip(ctx_embs, ctx_texts, ctx_edges_list):
                s = cosine_sim(emb, ce)

                # --- EXACT ENTITY MATCH PENALTY (STRICT AND LOGIC) ---
                u_safe, v_safe = str(u).lower(), str(v).lower()
                cu_safe, cv_safe = str(cu).lower(), str(cv).lower()

                # Did it find a match for the Subject? (Checking both context subject and object)
                subj_match = (u_safe in cu_safe or cu_safe in u_safe or u_safe in cv_safe or cv_safe in u_safe)

                # Did it find a match for the Object? (Checking both context subject and object)
                obj_match = (v_safe in cv_safe or cv_safe in v_safe or v_safe in cu_safe or cu_safe in v_safe)

                # BOTH the subject and the object MUST match. If either is hallucinated, slash the score!
                if not (subj_match and obj_match):
                    s = s * 0.2  # Slashed by 80% to absolutely guarantee it fails the threshold
                # -----------------------------------------------------

                if s > best_cos:
                    best_cos, best_match = s, ct

        # ── Signal 2: NLI on the SPECIFIC CLAIM ────────
        # Evaluates just the specific triplet claim against the context.
        nli = nli_score(context, edge_text)

        # ── Hybrid score: high = likely hallucinated ─────────
        hybrid = 0.6 * (1.0 - best_cos) + 0.4 * nli

        rec = {
            "subject": u, "predicate": data["predicate"], "object": v,
            "claim_text": edge_text,
            "similarity_score": round(best_cos, 4),
            "nli_score": round(nli, 4),
            "hybrid_score": round(hybrid, 4),
            "best_context_match": best_match
        }
        (supported if hybrid < threshold else unsupported).append(rec)

    total = len(supported) + len(unsupported)
    score = round(len(unsupported) / total, 4) if total else 0.0

    return {
        "hallucination_score": score,
        "unsupported_claims":  unsupported,
        "supported_claims":    supported,
        "context_triplets":    ctx_triplets,
        "answer_triplets":     ans_triplets,
        "context_graph":       G_ctx,
        "answer_graph":        G_ans,
        "total_claims":        total
    }

print("✅ Member 2 module ready")